# 05 — Integration test: train, approve, test and predict

Updated: 2026-11-01.

After notebook 02, Run All to test every deployed model in this namespace.
Set `test_csv_input` during deployment, or supply each model's test CSV in
`CSV_INPUTS` below. Training, test and inference data are selected separately;
this notebook does not create a split or substitute training data for test data.

For each model: preliminary training → verify metadata → full training with
that metadata → approve both model and Development status → run testing and
inference in parallel → verify outputs. Two models produce **eight executions**.

The run cell starts SageMaker jobs and approves this test's verified full models.
Use your development namespace and keep the kernel running for subsequent phases.
Reusing the same label and inputs resumes saved executions; use a new label after
redeployment. Existing jobs are not cancelled on a failure.

This checks the normal Processing paths and their prediction and postprocessing files.
HPO, native Batch Transform, drift and MLflow finalization have separate tests.


## Load the deployed application

- Locate this checkout and read the namespace/publication saved by notebook 02.
- Import the integration helper and ordinary AWS SDK; no model libraries are loaded.
- Prepare the display utilities without starting jobs or changing approvals.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import HTML, display
from html import escape

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
for directory in (ROOT / "notebooks/source/05_integration_test", ROOT / "mlops-models/src"):
    sys.path.insert(0, str(directory))
from integration_checks import IntegrationSuite

config_path = ROOT / "notebooks/config.json"
config = json.loads(config_path.read_text()) if config_path.is_file() else {}
namespace = config.get("namespace", "")
publication_path = ROOT / ".runs/source-deploy" / namespace / "publication.json"
publication = json.loads(publication_path.read_text()) if namespace and publication_path.is_file() else {}
print("Namespace:", namespace or "set NAMESPACE below")
print("Publication:", publication_path if publication else "provide PIPELINES below")

## Select inputs

- Keep MODEL_IDS and PIPELINES empty to use every deployed model and the saved three pipelines.
- Leave CSV_INPUTS empty only when the deployed pipelines already have the required data defaults.
- Set `prediction_test` to a separate S3 CSV for each model if testing has no deployed default.
- Reuse RUN_LABEL to resume this test; change it for a fresh test or after redeployment.

Training and inference overrides are independent. Editing config.json does not
change deployed defaults: rerun notebook 02, or override CSV_INPUTS here.
Feature Store deployments keep their deployed bindings when CSV_INPUTS is empty.


In [ ]:
RUN_LABEL = "three-pipelines-001"  # Same label resumes; a new label starts a fresh test.

# Optional overrides. Leave these unchanged for the complete default test.
MODEL_IDS = []                 # All deployed models.
PIPELINES = {}                 # Training + testing + inference saved by notebook 02.
CSV_INPUTS = {}                # Each pipeline's deployed input.
# Example: CSV_INPUTS = {"usecase1": {"training": "s3://your-bucket/labelled.csv"}}

NAMESPACE = namespace
AWS_REGION = publication.get("region", "")
MODEL_CODE_SOURCE = "artifact"  # "pipeline" tests the published inference code instead.
WAIT_TIMEOUT_MINUTES = 120


## Inspect the matrix before starting

- Read all three deployed definitions, pin their versions and validate the selected paths.
- Confirm that all belong to this namespace and use the Development registry workflow.
- Save the test configuration and show the planned executions; no jobs start here.

In [ ]:
suite = IntegrationSuite(
    ROOT, publication, run_label=RUN_LABEL, model_ids=MODEL_IDS,
    csv_inputs=CSV_INPUTS, pipelines=PIPELINES, namespace=NAMESPACE, region=AWS_REGION,
    model_code_source=MODEL_CODE_SOURCE, timeout_minutes=WAIT_TIMEOUT_MINUTES)
plan = suite.prepare()
for kind, pipeline in suite.pipelines.items():
    print(kind, pipeline["arn"], "version", pipeline["version"])
print("Planned executions:", len(plan))
print("Evidence:", suite.directory)
display(pd.DataFrame(plan))

## Run the integration test

- Start each parallel phase, wait for completion and verify its required artifacts.
- Approve both fields on the verified full models, then run testing and inference for each model.
- Refresh progress and save a final check table even when a phase fails.

**Effect:** this cell launches jobs and approves the exact full models created by
this test. The role needs pipeline execution, job inspection, S3 read/list and model
package update permissions. The 120-minute setting bounds waiting, not job runtime.
Failures and timeouts leave existing jobs/resources available for investigation.

In [ ]:
progress = display(HTML("Starting integration test…"), display_id=True)

def show_progress(report):
    columns = ["model", "phase", "status", "checks", "active_steps", "minutes", "cached_steps"]
    rows = pd.DataFrame(report["executions"], columns=columns)
    progress.update(HTML(
        f"<p><b>{escape(report['status'])}</b> · {report['elapsed_minutes']:.1f} minutes · "
        f"{report['passed_checks']} checks passed · {report['failed_checks']} failed</p>"
        + rows.to_html(index=False)))

suite.on_update = show_progress
try:
    suite.run()
finally:
    result = suite.report()
    print("Result:", result["status"], "| elapsed minutes:", result["elapsed_minutes"])
    if result["error"]:
        print("Failure:", result["error"])
    display(pd.DataFrame(result["checks"]))
    print("Saved evidence:", suite.directory / "results.json")

## Inspect executions and outputs

- Display the exact execution ARNs, registered model versions and S3 locations.
- Inspect the saved results after completion or a failure without starting another job.
- Use notebook 04 with an execution ARN for individual execution and step status.

A passed result means the selected normal paths produced the required files and
passed the recorded identity/metadata checks. It does not establish predictive
quality, quota headroom or the behavior of configurations outside this matrix.
Changing code and redeploying requires a new RUN_LABEL to test the new versions.

In [ ]:
result = suite.report()
display(pd.DataFrame(result["executions"]))
print("Checks:", result["passed_checks"], "passed,", result["failed_checks"], "failed")
print("State for resume:", suite.path)